In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 07. MoE Load Balancing Loss | MoE 进阶：负载均衡损失函数 (Load Balancing Loss)

在真实的 MoE 模型（如 Mixtral 8x7B, DeepSeek）训练中，会遇到一个非常严重的问题：**路由崩塌 (Router Collapse)**。
即门控网络“偷懒”，把所有的 Token 都发给了第 0 号和第 1 号专家，导致其他专家被饿死（闲置），不仅失去了 MoE 的意义，还会导致算力非常不均衡（OOM）。

更详细的解释：
路由崩塌源于随机初始化时，第0、1号专家在门控网络中的logit存在微乎其微的初始正向偏差，经Softmax指数函数放大后获得略高选中概率；在Top-2稀疏激活下，这两个专家因频繁接收Token而获得更多梯度更新，参数迅速强于他人，门控网络为降低主任务Loss进一步抬高其logit，形成“富者愈富”的正反馈闭环。同时，主任务Loss仅关注最终输出总和，对专家ID排列不变，故毫无动力探索其他专家，导致它们因梯度近零而永久停滞于初始化状态。最终，这两个初始的“幸运儿”形成强耦合垄断，造成有效参数量坍缩、其余专家饿死，并引发分布式训练中指定GPU显存过载（OOM）。

解决方法：**如何用代码实现 MoE 的辅助损失函数 (Auxiliary Loss) 来强制负载均衡？**

为了让 $N$ 个 Token 均匀地分配给 $E$ 个专家，我们需要设计一个惩罚项，加到总的 CrossEntropy Loss 里。
Mixtral / Switch Transformer 使用的经典公式：
$$ L_{aux} = \alpha \cdot E \sum_{i=1}^E f_i \cdot P_i $$

- $E$: 专家总数。
- $f_i$: 专家 $i$ 被路由到的 **Token 比例**（即选了专家 $i$ 的 token 数 / 总 token 数）。
- $P_i$: 专家 $i$ 在所有 Token 上的 **平均路由概率得分**（Softmax 之后的概率的均值）。
- $\alpha$: 辅助损失的权重系数（通常很小，如 0.01）。

**为什么这个公式有效？**
根据均值不等式，给定总和为 1 的 $f$ 和 $P$，当且仅当所有的 $f_i = 1/E$ 且 $P_i = 1/E$ 时（即绝对均匀分配），它们的内积（点积）之和最小。优化器为了降低这个 Loss，会拼命把 Token 往不同的专家那里赶！

# MoE 负载均衡损失（Aux Loss）手算推导实例

本实例基于 `top_k = 2`，演示代码中的 `P_i`、`f_i` 是如何算出来的，以及未选中的 Softmax 概率为什么被丢弃。

---

## 1. 实验参数设定

| 参数 | 值 | 说明 |
| :--- | :--- | :--- |
| 专家总数 $E$ | **3** | 专家 0、专家 1、专家 2 |
| Token 总数 $T$ | **2** | Token A、Token B |
| 每个 Token 选专家数 $K$ | **2** | Top-2 路由 |
| 辅助损失权重 $\alpha$ | **0.01** | 固定超参数 |
| 总路由槽位数 | $T \times K = \mathbf{4}$ | 代码中归一化使用的关键分母 |

---

## 2. 原始路由数据（Softmax 输出）

每个 Token 的真实 Softmax 概率分布如下（加粗项为被选中的 Top-2）：

| Token | Top-1 (选中) | Top-2 (选中) | 未选中 (丢弃) |
| :--- | :--- | :--- | :--- |
| **Token A** | 专家 0 : **0.7** | 专家 1 : **0.2** | 专家 2 : 0.1 |
| **Token B** | 专家 1 : **0.6** | 专家 2 : **0.3** | 专家 0 : 0.1 |

> **注意**：未选中的 0.1 虽然属于 Softmax 的一部分，但在代码实现中**完全不参与** Loss 计算。

---

## 3. 计算平均路由概率 $P_i$

**第 1 步**：累加每个专家被选中的原始概率（`scatter_add`）

- 专家 0：收到 Token A 的 `0.7` → 总分 = **0.7**
- 专家 1：收到 Token A 的 `0.2` + Token B 的 `0.6` → 总分 = **0.8**
- 专家 2：收到 Token B 的 `0.3` → 总分 = **0.3**

**第 2 步**：除以总槽位数 $T \times K = 4$（代码中的归一化）

$$ P_i = \frac{\text{累加总分}}{4} $$

| 专家 | 计算公式 | 结果 $P_i$ |
| :--- | :--- | :--- |
| 专家 0 | $0.7 / 4$ | **0.175** |
| 专家 1 | $0.8 / 4$ | **0.2** |
| 专家 2 | $0.3 / 4$ | **0.075** |

---

## 4. 计算 Token 分配比例 $f_i$

**第 1 步**：统计每个专家被选中的次数

- 专家 0：被 Token A 选中（1次） → 次数 = **1**
- 专家 1：被 Token A、B 选中（2次） → 次数 = **2**
- 专家 2：被 Token B 选中（1次） → 次数 = **1**

**第 2 步**：除以总槽位数 $T \times K = 4$

$$ f_i = \frac{\text{选中次数}}{4} $$

| 专家 | 计算公式 | 结果 $f_i$ |
| :--- | :--- | :--- |
| 专家 0 | $1 / 4$ | **0.25** |
| 专家 1 | $2 / 4$ | **0.5** |
| 专家 2 | $1 / 4$ | **0.25** |

> **校验**：$f_0 + f_1 + f_2 = 0.25 + 0.5 + 0.25 = 1.0$，完美归一化。

---

## 5. 计算最终的 Auxiliary Loss

代入公式：

$$ L_{aux} = \alpha \cdot E \cdot \sum_{i=1}^E f_i \cdot P_i $$

**第 1 步**：计算每个专家的乘积 $f_i \cdot P_i$

| 专家 | $f_i$ | $\times$ | $P_i$ | $=$ | 乘积 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 专家 0 | 0.25 | $\times$ | 0.175 | $=$ | **0.04375** |
| 专家 1 | 0.5 | $\times$ | 0.2 | $=$ | **0.1** |
| 专家 2 | 0.25 | $\times$ | 0.075 | $=$ | **0.01875** |

**第 2 步**：求和

$$ \sum f_i P_i = 0.04375 + 0.1 + 0.01875 = \mathbf{0.1625} $$

**第 3 步**：乘以 $\alpha$ 和 $E$

$$ L_{aux} = 0.01 \times 3 \times 0.1625 = \mathbf{0.004875} $$

---

## 6. 核心答疑：为什么丢弃未选中的 Softmax（0.1）？

| 对比维度 | 如果强行加入未选中的 0.1 | 代码实际做法（丢弃） |
| :--- | :--- | :--- |
| **统计范围** | 所有专家的 Softmax 输出 | 仅限被选中的 Top-K 槽位 |
| **对专家 2 的影响** | 虽然没干活，但 $P_2$ 会因 0.1 而增大 | 完全忽略，$P_2$ 只来自被选中的 0.3 |
| **梯度方向** | 优化器被迫压低“未干活”专家的概率（浪费算力） | 只惩罚“实际过载”的专家（精准打击） |
| **工程含义** | 论文公式的字面定义（模糊） | **被选中槽位上的条件平均概率**（务实） |

**结论**：未选中的概率没有参与前向计算（不消耗 FLOPs），强行加入只会引入噪声，干扰主任务的梯度更新。代码中的 $P_i$ 本质上是**“专家在被激活时给出的平均置信度”**，而非“全局平均”。

In [ ]:
def compute_load_balancing_loss(
    routing_weights: torch.Tensor, 
    selected_experts: torch.Tensor, 
    num_experts: int, 
    top_k: int,
    alpha: float = 0.01
):
    """
    计算 MoE 的负载均衡辅助损失（支持 Top-K 路由）
    
    Args:
        routing_weights: [batch_size * seq_len, top_k]，每个 token 选中的 K 个专家的权重（已归一化）
        selected_experts: [batch_size * seq_len, top_k]，每个 token 选中的 K 个专家的索引
        num_experts: 专家总数 E
        top_k: 每个 token 选择的专家数量 K
        alpha: 损失权重系数
    
    Returns:
        aux_loss: 标量，负载均衡损失
    """
    batch_size_x_seq_len, _ = selected_experts.shape
    total_tokens = batch_size_x_seq_len
    
    # ==========================================
    # TODO 1: 计算 P_i（每个专家的平均路由概率得分）
    # ==========================================
    P_i = torch.zeros(num_experts, dtype=routing_weights.dtype, device=routing_weights.device)
    P_i.scatter_add_(0, selected_experts.flatten(), routing_weights.flatten())
    P_i = P_i / (total_tokens * top_k)
    
    # ==========================================
    # TODO 2: 计算 f_i（每个专家实际分到的 Token 比例）
    # ==========================================
    expert_mask = F.one_hot(selected_experts, num_classes=num_experts)
    tokens_per_expert = expert_mask.sum(dim=(0, 1)).float()
    f_i = tokens_per_expert / (total_tokens * top_k)
    
    # ==========================================
    # TODO 3: 计算最终的 auxiliary loss
    # ==========================================
    aux_loss = alpha * num_experts * (f_i * P_i).sum()
                                                                                                                                                                                  
    return aux_loss


**工程要点**

- **Top-K 兼容性**：代码支持任意 K 值，通过 `(total_tokens * top_k)` 归一化确保比例计算正确。
- **数值稳定性**：使用 `scatter_add_` 而非循环累加，提升计算效率和数值稳定性。
- **超参数调优**：$\alpha$ 通常设为 0.01，过大会影响主任务性能，过小则无法有效平衡负载。
- **与主损失结合**：在实际训练中，将 `aux_loss` 加到 CrossEntropy Loss 上：`total_loss = ce_loss + aux_loss`。